In [ ]:
import pandas as pd

ex_campaignmetrics = pd.read_csv("https://cdn.enqurious.com/documents/2443026c-e6fe-4dde-b723-de45095f4634_excampaignmetrics.csv")

ex_campaigns = pd.read_csv("https://cdn.enqurious.com/documents/2fb52f3f-77ae-4464-bd5c-a7753bee1fdb_excampaigns.csv")

ex_channel = pd.read_csv("https://cdn.enqurious.com/documents/2faf9174-ef92-43f7-bc6d-bf5e9b2ede62_exchannel.csv")

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# ==============================================================================
# STEP 1: Load and Prepare the Data_
# ==============================================================================
# Create copies of the original DataFrames so the source data remains unchanged.
campaigns = ex_campaigns.copy()
channel = ex_channel.copy()
metrics = ex_campaignmetrics.copy()

# Convert date columns from string format to datetime.
# This allows date calculations such as campaign duration.
campaigns['start_date'] = pd.to_datetime(campaigns['start_date'])
campaigns['end_date'] = pd.to_datetime(campaigns['end_date'])
metrics['metric_date'] = pd.to_datetime(metrics['metric_date'])

# ==============================================================================
# STEP 2: Aggregate Daily Metrics for Each Campaign
# ==============================================================================
# Combine all daily records into a single summary row per campaign.
campaign_agg = (
    metrics
    .groupby('campaign_id', as_index=False)
    .agg(
        total_impressions=('impressions', 'sum'),
        total_clicks=('clicks', 'sum'),
        total_conversions=('conversions', 'sum'),
        total_bounces=('bounces', 'sum'),

        # Count the number of unique dates to calculate average daily conversions.
        distinct_dates=('metric_date', 'nunique')
    )
)

# Calculate the average number of conversions generated per day.
campaign_agg['avg_daily_conversions'] = (
    campaign_agg['total_conversions'] /
    campaign_agg['distinct_dates']
).round(2)

# The distinct_dates column has served its purpose, so remove it.
campaign_agg.drop(columns='distinct_dates', inplace=True)

# ==============================================================================
# STEP 3: Combine Campaign Summary with Campaign and Channel Details
# ==============================================================================
# Merge aggregated metrics with campaign information,
# then attach the corresponding channel information.
df = (
    campaign_agg
    .merge(campaigns, on='campaign_id')
    .merge(channel, on='channel_id')
)

# ==============================================================================
# STEP 4: Create Additional Business Metrics
# ==============================================================================

# Calculate the total duration of each campaign.
# +1 is added so both the start and end dates are included.
df['campaign_duration_days'] = (
    df['end_date'] - df['start_date']
).dt.days + 1

# Calculate Click Through Rate (CTR).
# CTR = (Clicks / Impressions) × 100
# np.where() prevents division by zero.
df['click_through_rate'] = np.where(
    df['total_impressions'] > 0,
    (df['total_clicks'] / df['total_impressions'] * 100).round(2),
    0
)

# Calculate Conversion Rate.
# Conversion Rate = (Conversions / Clicks) × 100
# Again, handle campaigns with zero clicks safely.
df['conversion_rate'] = np.where(
    df['total_clicks'] > 0,
    (df['total_conversions'] / df['total_clicks'] * 100).round(2),
    0
)

# Format percentage values for easier readability.
df['click_through_rate'] = df['click_through_rate'].astype(str) + '%'
df['conversion_rate'] = df['conversion_rate'].astype(str) + '%'

# ==============================================================================
# STEP 5: Apply Window Ranking Functions Within Each Marketing Channel
# ==============================================================================

# RANK()
# Assigns rankings based on total conversions.
# Campaigns with the same conversions receive the same rank,
# and the next rank is skipped.
df['channel_rank'] = (
    df.groupby('channel_name')['total_conversions']
      .rank(method='min', ascending=False)
      .astype(int)
)

# DENSE_RANK()
# Similar to RANK(), but does not skip ranking numbers after ties.
df['dense_channel_rank'] = (
    df.groupby('channel_name')['total_conversions']
      .rank(method='dense', ascending=False)
      .astype(int)
)

# ROW_NUMBER()
# Assigns a unique sequential number to every campaign,
# even when conversion counts are identical.
df['row_num'] = (
    df.groupby('channel_name')['total_conversions']
      .rank(method='first', ascending=False)
      .astype(int)
)

# ==============================================================================
# STEP 6: Select the Required Columns and Sort the Final Output
# ==============================================================================
result = (
    df[[
        'campaign_id',
        'channel_name',
        'campaign_type',
        'budget',
        'total_impressions',
        'total_clicks',
        'total_conversions',
        'total_bounces',
        'avg_daily_conversions',
        'click_through_rate',
        'conversion_rate',
        'campaign_duration_days',
        'channel_rank',
        'dense_channel_rank',
        'row_num'
    ]]
    # Display campaigns ranked within each marketing channel.
    .sort_values(['channel_name', 'channel_rank'])
    .reset_index(drop=True)
)

# ==============================================================================
# STEP 7: Display the Final Report
# ==============================================================================
display(result)

,campaign_id,channel_name,campaign_type,budget,total_impressions,total_clicks,total_conversions,total_bounces,avg_daily_conversions,click_through_rate,conversion_rate,campaign_duration_days,channel_rank,dense_channel_rank,row_num
0,259,Affiliate Marketing,Conversion Campaign,27770.0,266786,20627,11588,8578,772.53,7.73%,56.18%,192,1,1,1
1,1539,Affiliate Marketing,Flash Sale,5524.0,197695,17213,11255,8664,803.93,8.71%,65.39%,344,2,2,2
2,255,Affiliate Marketing,Seasonal Sale,21610.0,234574,17160,11199,7158,746.60,7.32%,65.26%,115,3,3,3
3,752,Affiliate Marketing,Engagement Campaign,12414.0,225512,16933,10423,10664,744.50,7.51%,61.55%,135,4,4,4
4,364,Affiliate Marketing,Retargeting,14424.0,220588,16395,10273,10325,684.87,7.43%,62.66%,266,5,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1493,482,SMS Marketing,First-Time Buyer Campaign,14710.0,64244,4041,1546,1456,309.20,6.29%,38.26%,262,226,220,226
1494,563,SMS Marketing,Engagement Campaign,16686.0,90135,4455,1481,1680,211.57,4.94%,33.24%,253,227,221,227
1495,609,SMS Marketing,Product Launch,29580.0,75815,5547,1479,1438,246.50,7.32%,26.66%,178,228,222,228
1496,1165,SMS Marketing,Discount,10196.0,69607,5000,1151,3033,230.20,7.18%,23.02%,321,229,223,229


In [ ]:
import pandas as pd

ex_campaignmetrics = pd.read_csv("https://cdn.enqurious.com/documents/2443026c-e6fe-4dde-b723-de45095f4634_excampaignmetrics.csv")

ex_campaigns = pd.read_csv("https://cdn.enqurious.com/documents/2fb52f3f-77ae-4464-bd5c-a7753bee1fdb_excampaigns.csv")

ex_channel = pd.read_csv("https://cdn.enqurious.com/documents/2faf9174-ef92-43f7-bc6d-bf5e9b2ede62_exchannel.csv")

In [ ]:
import pandas as pd
from IPython.display import display

# ==============================================================================
# STEP 1: Load and Prepare the Data
# ==============================================================================
# Create a copy of the campaign metrics dataset to avoid modifying the original.
metrics = ex_campaignmetrics.copy()

# Convert the metric_date column to datetime format.
# This ensures dates are sorted chronologically.
metrics['metric_date'] = pd.to_datetime(metrics['metric_date'])

# ==============================================================================
# STEP 2: Sort the Data
# ==============================================================================
# Arrange records by campaign and date.
# This is necessary before performing cumulative and running calculations.
metrics_sorted = (
    metrics
    .sort_values(['campaign_id', 'metric_date'])
    .reset_index(drop=True)
)

# ==============================================================================
# STEP 3: Calculate Cumulative Conversions
# ==============================================================================
# For each campaign, keep a running total of conversions.
# cumsum() continuously adds each day's conversions to the previous total.
metrics_sorted['cumulative_conversions'] = (
    metrics_sorted
    .groupby('campaign_id')['conversions']
    .cumsum()
)

# ==============================================================================
# STEP 4: Calculate Running Average of Clicks
# ==============================================================================
# Compute the average number of clicks from the first day
# up to the current day for each campaign.
# expanding() gradually increases the calculation window.
metrics_sorted['running_avg_clicks'] = (
    metrics_sorted
    .groupby('campaign_id')['clicks']
    .expanding()
    .mean()
    .round(2)
    .reset_index(level=0, drop=True)
)

# ==============================================================================
# STEP 5: Select and Format the Final Output
# ==============================================================================
# Keep only the required columns and rename 'conversions'
# to make its meaning clearer in the final report.
daily_trend = (
    metrics_sorted[[
        'campaign_id',
        'metric_date',
        'conversions',
        'cumulative_conversions',
        'running_avg_clicks'
    ]]
    .rename(columns={
        'conversions': 'daily_conversions'
    })
)

# ==============================================================================
# STEP 6: Display the Results
# ==============================================================================
# Show the first 10 records of the final report.
# Each row displays:
# • Daily conversions
# • Cumulative conversions
# • Running average of clicks for that campaign
display(daily_trend.head(10))

,campaign_id,metric_date,daily_conversions,cumulative_conversions,running_avg_clicks
0,101,2017-09-29,232,232,548.00
1,101,2017-09-30,80,312,564.00
2,101,2017-10-01,624,936,909.67
3,101,2017-10-02,396,1332,791.00
4,101,2017-10-03,1391,2723,950.80
5,101,2017-10-04,204,2927,1028.67
6,101,2017-10-05,107,3034,1050.71
7,101,2017-10-06,577,3611,1059.50
8,101,2017-10-07,1466,5077,1143.67
9,101,2017-10-08,110,5187,1048.60
